In [0]:

cosmos_uri = "https://azcosmosdb2239.documents.azure.com:443/"
cosmos_key = "FriSO3pQve3g8PEBQ10fGs0qtsbz1teKP5TqstBD9hfn8R8hRjlGY5YBBMHgk6NmdeDPPbXELxQSACDbEqRYAA=="
database_name = "DatabaseAdf"
container_name = "MyContainer"


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CosmosDBIntegration") \
    .config("spark.cosmos.accountEndpoint", cosmos_uri) \
    .config("spark.cosmos.accountKey", cosmos_key) \
    .config("spark.cosmos.database", database_name) \
    .config("spark.cosmos.container", container_name) \
    .getOrCreate()

In [0]:
spark

In [0]:
# Read data from Cosmos DB
df = spark.read \
    .format("cosmos.oltp") \
    .option("spark.cosmos.accountEndpoint", cosmos_uri) \
    .option("spark.cosmos.accountKey", cosmos_key) \
    .option("spark.cosmos.database", database_name) \
    .option("spark.cosmos.container", container_name) \
    .load()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

df = df.dropDuplicates()
df = df.filter(df.MSRP > 10000) 
df = df.withColumn("Price_Per_Horsepower", df.MSRP / df.Horsepower)
df = df.withColumn("Make", upper(df.Make)).withColumn("Model", upper(df.Model))
df = df.withColumnRenamed("MSRP", "Price").withColumnRenamed("EngineSize", "Engine_Size")




In [0]:
df.show()

+----------+---------+----------+-----+------+-------------+------+---------+------+--------------------+--------------------+-------+--------+-----------+-----------+------+--------------------+
|DriveTrain|Cylinders|Horsepower|Price|Length|         Make|Origin|Wheelbase|Weight|               Model|                  id|Invoice|MPG_City|Engine_Size|MPG_Highway|  Type|Price_Per_Horsepower|
+----------+---------+----------+-----+------+-------------+------+---------+------+--------------------+--------------------+-------+--------+-----------+-----------+------+--------------------+
|     Front|        4|       126|14740|   178|       NISSAN|  Asia|      100|  2581|    SENTRA 1.8 S 4DR|026eab84-cf53-425...|13747.0|      28|        1.8|         35| Sedan|  116.98412698412699|
|     Front|        4|       130|15460|   168|         FORD|   USA|      103|  2606|        FOCUS SE 4DR|2c0a04f0-61e3-4b6...|14496.0|      26|        2.0|         33| Sedan|  118.92307692307692|
|     Front|        

In [0]:
# Azure Blob Storage configurations
storage_account_name = "azureblobstorage2239"
storage_account_key = "8HUbM56zpnqtFXmtue0bYWDG48l86vIB/ENZf0GSWrSCY/lBzO9bgClHqOQv7GMvmEnBxXwIBvvv+ASthXfkyg=="
container_name = "databricks-output"
output_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/output_data"


In [0]:
# Set the storage account key
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", storage_account_key)


In [0]:
df.write.mode("overwrite").option("header","true").csv(output_path)